# Prepare Rhino training dataset

Wrapper over local scripts:
- `scripts/download_sources.py` — McNeel [rhino-developer-samples](https://github.com/mcneel/rhino-developer-samples) + developer.rhino3d.com guides
- `scripts/build_dataset.py` — format chat JSONL
- `scripts/make_synthetic.py` — template augmentation

Run locally or in Colab (CPU runtime is fine).

In [ ]:
import os
from pathlib import Path

REPO = Path("/content/My-LLM") if Path("/content/My-LLM").exists() else Path.cwd()
if str(REPO).startswith("/content") and not (REPO / "scripts").exists():
    !git clone https://github.com/YOUR_USER/My-LLM.git /content/My-LLM
    REPO = Path("/content/My-LLM")

os.chdir(REPO)
print("Working directory:", REPO)

In [ ]:
!pip install -q -r requirements.txt

## 1. Seed data (immediate training)

The repo ships ~52 curated examples. Regenerate if needed:

In [ ]:
!python scripts/init_seed.py

## 2. Download public sources

Fetches **mcneel/rhino-developer-samples** (zip) and polite doc pages from developer.rhino3d.com.

In [ ]:
# Full download (may take a few minutes)
!python scripts/download_sources.py

# Or samples only:
# !python scripts/download_sources.py --samples-only

# Or docs only with limit:
# !python scripts/download_sources.py --docs-only --max-pages 10

In [ ]:
from pathlib import Path
import json

manifest = Path("data/raw/download_manifest.json")
if manifest.exists():
    print(json.dumps(json.loads(manifest.read_text()), indent=2))
else:
    print("Run download_sources.py first")

## 3. Build JSONL

Merges samples + docs into `data/processed/rhino_sft.jsonl`, dedupes by assistant hash.

In [ ]:
!python scripts/build_dataset.py --append-seed

## 4. Synthetic augmentation (toward 3k+)

In [ ]:
!python scripts/make_synthetic.py --count 100

In [ ]:
import json
from collections import Counter
from pathlib import Path

path = Path("data/processed/rhino_sft.jsonl")
rows = [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]
langs = Counter(r["meta"]["lang"] for r in rows)
print(f"Total rows: {len(rows)}")
print("By lang:", dict(langs))
print("\nSample:")
print(json.dumps(rows[0], indent=2)[:600])

## 5. Upload to Drive for training

Copy `rhino_sft.jsonl` to `/content/drive/MyDrive/rhino_llm/` then open **02_train_rhino_llm.ipynb**.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    import shutil
    dest = Path("/content/drive/MyDrive/rhino_llm")
    dest.mkdir(parents=True, exist_ok=True)
    shutil.copy("data/processed/rhino_sft.jsonl", dest / "rhino_sft.jsonl")
    print("Copied to", dest)
except ImportError:
    print("Not in Colab — upload rhino_sft.jsonl manually before training.")

## Expanding toward 3k+ examples

1. Re-run `download_sources.py` after network access to McNeel GitHub.
2. Add URLs to `DOC_SEED_URLS` in `download_sources.py`.
3. Increase `make_synthetic.py --count`.
4. Record new sources in `data/SOURCES.md`.

Each row format:
```json
{"messages": [{"role": "user", ...}, {"role": "assistant", ...}], "meta": {"lang": "python|csharp|concept", "topic": "..."}}
```